# 05 · Forecasting

Modelamos y pronosticamos la serie de la canción **viral** (streams diarios). El
notebook es **autónomo**: arma la serie en memoria desde `data/charts_argentina.csv`,
sin depender de que el 03 se haya corrido antes.

## 0. Armado de la serie (en memoria)

In [ ]:
import sys, logging
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.utils import (
    construir_serie, interpolar_serie, graficar_serie, resumen_estadistico,
    calcular_fft, graficar_espectro, descomponer_MA7_MA21, energia_serie,
    test_estacionariedad, ajustar_arima, ajustar_prophet,
)

# Prophet es verboso: bajamos el logging
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

# --- Parámetro editable ---
CANCION_VIRAL = "C.R.O: Bzrp Music Sessions, Vol. 29"
REGION        = "Argentina"
CHART         = "top200"
# --------------------------

df = pd.read_csv("../data/charts_argentina.csv", parse_dates=["date"])
cruda = construir_serie(df, {"title": CANCION_VIRAL, "region": REGION, "chart": CHART,
                             "valor": "streams", "agregado": "sum"})
serie = interpolar_serie(cruda)
print(f"{CANCION_VIRAL}: {len(serie)} días | {len(serie) - cruda.index.nunique()} huecos "
      f"| {serie.index.min().date()} -> {serie.index.max().date()}")

def pred_prophet_safe(train, h):
    """Pronóstico Prophet de h pasos; devuelve None si el backend de Stan no carga
    en el entorno (así el notebook no se corta y sigue con Naive y ARIMA)."""
    try:
        mp = ajustar_prophet(train)
        return mp.predict(mp.make_future_dataframe(periods=h, freq="D"))["yhat"].to_numpy()[-h:]
    except Exception as e:
        print(f"[aviso] Prophet no disponible ({type(e).__name__}); se omite en este backtest.")
        return None

## 1. Exploratorio

In [ ]:
graficar_serie(serie, f"Streams diarios - {CANCION_VIRAL}")
plt.show()
print("resumen:", {k: round(v) for k, v in resumen_estadistico(serie).items()})

## 2. Fourier

In [ ]:
frecs, mags = calcular_fft(serie)
graficar_espectro(frecs, mags, f"Espectro - {CANCION_VIRAL}")
plt.axvline(1/7, color="red", ls="--", lw=1, label="1/7 (semanal)")
plt.legend(); plt.show()
f_pico = frecs[np.argmax(mags)]
print(f"Frecuencia dominante: {f_pico:.4f} ciclos/día -> período {1/f_pico:.1f} días")

## 3. Filtrado (MA7 / MA21)

In [ ]:
desc = descomponer_MA7_MA21(serie)
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
ax[0].plot(serie.index, serie.values); ax[0].set_title("Original")
ax[1].plot(desc["tendencia"]); ax[1].set_title("Tendencia (MA21)")
ax[2].plot(desc["estacional"]); ax[2].set_title("Estacional (MA7 − MA21)")
ax[3].plot(desc["residuo"]); ax[3].set_title("Residuo (serie − MA7)")
for a in ax: a.set_ylabel("streams")
plt.tight_layout(); plt.show()

## 4. Energía y Parseval

In [ ]:
filtrada = desc["tendencia"] + desc["estacional"]
e_antes = energia_serie(serie)
e_desp  = energia_serie(filtrada)
print(f"ANTES  : E_t={e_antes['energia_tiempo']:.3e} E_f={e_antes['energia_frecuencia']:.3e} Parseval={e_antes['parseval_ok']}")
print(f"DESPUÉS: E_t={e_desp['energia_tiempo']:.3e} E_f={e_desp['energia_frecuencia']:.3e} Parseval={e_desp['parseval_ok']}")
print(f"Energía retenida: {100 * e_desp['energia_tiempo'] / e_antes['energia_tiempo']:.2f}%")

## 5. Estacionariedad → orden de diferenciación `d`

Testeamos la serie; si no es estacionaria, diferenciamos una vez y volvemos a testear.
El primer `d` que la vuelve estacionaria es el que usaremos en el ARIMA.

In [ ]:
est0 = test_estacionariedad(serie)
print("d=0  ADF:", est0["adf"]["p_valor"], "estac:", est0["adf"]["estacionaria"],
      "| KPSS:", est0["kpss"]["p_valor"], "estac:", est0["kpss"]["estacionaria"])

serie_diff = serie.diff().dropna()
est1 = test_estacionariedad(serie_diff)
print("d=1  ADF:", est1["adf"]["p_valor"], "estac:", est1["adf"]["estacionaria"],
      "| KPSS:", est1["kpss"]["p_valor"], "estac:", est1["kpss"]["estacionaria"])

d = 0 if est0["adf"]["estacionaria"] else 1
print(f"-> se usa d = {d}")

## 6. ACF y PACF → elección MANUAL de `(p, d, q)`

Miramos la ACF y la PACF de la serie **diferenciada** (`d=1`) y elegimos a mano dos
candidatos, según dónde "cortan":
- **PACF** corta después del lag *p* → sugiere el orden **AR(p)**.
- **ACF** corta después del lag *q* → sugiere el orden **MA(q)**.

No hacemos ningún barrido automático.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(serie_diff, lags=30, ax=ax[0]); ax[0].set_title("ACF (serie diferenciada)")
plot_pacf(serie_diff, lags=30, ax=ax[1], method="ywm"); ax[1].set_title("PACF (serie diferenciada)")
plt.tight_layout(); plt.show()

**Lectura y candidatos.** Sobre la serie diferenciada, la PACF muestra un corte
temprano (uno o dos rezagos significativos → componente **AR** de orden 1-2) y la ACF
decae rápido con un primer rezago marcado (componente **MA** de orden ~1). Con eso
elegimos dos combinaciones razonables:

- **(1, 1, 1)** - el modelo más parsimonioso: un AR y un MA sobre la serie diferenciada.
- **(2, 1, 1)** - permite un rezago autorregresivo extra por si la PACF pide orden 2.

## 7. Ajuste ARIMA de los dos candidatos y elección por AIC

In [ ]:
candidatos = [(1, 1, 1), (2, 1, 1)]
resultados = {}
for order in candidatos:
    r = ajustar_arima(serie, order)
    resultados[order] = r
    print(f"ARIMA{order}: AIC = {r['aic']:.1f} | BIC = {r['bic']:.1f}")

order_elegido = min(resultados, key=lambda o: resultados[o]["aic"])
print(f"-> elegido por menor AIC: ARIMA{order_elegido}")

**Por qué esa.** Nos quedamos con el orden de **menor AIC** (impreso arriba): el AIC
penaliza la complejidad, así que si el candidato con un parámetro extra no reduce el AIC
lo suficiente, gana el más simple. La celda deja registrado cuál ganó con los datos
reales.

## 8. Ajuste Prophet (configuración por defecto)

In [ ]:
try:
    modelo_prophet = ajustar_prophet(serie)
    print("Prophet ajustado:", type(modelo_prophet).__name__)
except Exception as e:
    print(f"[aviso] Prophet no disponible ({type(e).__name__}); se omite el ajuste global.")

## 9. Backtest A - meseta final (últimos 25 días)

Entrenamos con todo menos los últimos **25 días** y predecimos ese tramo con ambos
modelos (ARIMA con el orden elegido y Prophet) más un **baseline naive** (repetir el
último valor de train). Graficamos predicho vs. real y calculamos el MAE de cada uno.

In [ ]:
H = 25
serie_train, serie_test = serie.iloc[:-H], serie.iloc[-H:]

# ARIMA: reajustamos el orden elegido solo con el train y pronosticamos H pasos
arima_train = ajustar_arima(serie_train, order_elegido)["modelo"]
pred_arima = np.asarray(arima_train.forecast(steps=H))

# Prophet: reajustamos con el train y pedimos H días futuros (seguro ante fallo de backend)
pred_prophet = pred_prophet_safe(serie_train, H)

# Baseline naive: repetir el último valor de entrenamiento durante los H días de test
pred_naive = np.repeat(serie_train.values[-1], H)

mae_arima   = float(np.mean(np.abs(pred_arima - serie_test.values)))
mae_naive   = float(np.mean(np.abs(pred_naive - serie_test.values)))
mae_prophet = float(np.mean(np.abs(pred_prophet - serie_test.values))) if pred_prophet is not None else float("nan")

plt.figure(figsize=(12, 5))
plt.plot(serie.index, serie.values, color="steelblue", label="Real", lw=1)
plt.plot(serie_test.index, pred_arima,   color="crimson", ls="--", marker="o", ms=3,
         label=f"ARIMA{order_elegido} (MAE {mae_arima:,.0f})")
if pred_prophet is not None:
    plt.plot(serie_test.index, pred_prophet, color="green", ls="--", marker="s", ms=3,
             label=f"Prophet (MAE {mae_prophet:,.0f})")
plt.plot(serie_test.index, pred_naive,   color="gray",    ls=":",  marker="^", ms=3,
         label=f"Naive (MAE {mae_naive:,.0f})")
plt.axvline(serie_test.index[0], color="gray", lw=0.8)
plt.title(f"Pronóstico últimos {H} días - {CANCION_VIRAL}")
plt.xlabel("Fecha"); plt.ylabel("streams"); plt.legend()
plt.tight_layout(); plt.show()

prophet_txt = f"{mae_prophet:,.0f}" if not np.isnan(mae_prophet) else "no disponible"
print(f"Rango real del test ({H} días): mín {serie_test.min():,.0f} | máx {serie_test.max():,.0f} "
      f"| amplitud {serie_test.max() - serie_test.min():,.0f}")
print("-" * 60)
print(f"MAE Naive (último valor): {mae_naive:,.0f}")
print(f"MAE ARIMA{order_elegido}       : {mae_arima:,.0f}")
print(f"MAE Prophet             : {prophet_txt}")

## 10. Backtest B - subida/pico y caída (donde SÍ hay dinámica)

El Backtest A cae sobre la **meseta** final, donde la serie ya está plana y hasta el
naive acierta. Para exigirle dinámica real a los modelos, hacemos un segundo backtest
sobre el tramo con más movimiento.

**Aclaración:** en esta canción el pico ocurre el **día 3** desde la entrada al chart
(2020-05-23), así que no existe una ventana de 20-30 días *antes* del pico. Entrenamos
entonces con los **primeros 20 días** (entrada + pico + inicio de la caída) y testeamos
sobre los **25 días siguientes**, que son de **caída empinada** (~106k → ~79k streams):
ahí sí hay una tendencia marcada que un buen modelo debería anticipar y el naive no.

In [ ]:
T_TRAIN = 20   # primeros 20 días: entrada, pico (día 3) e inicio de la caída
HB = 25        # test sobre la caída empinada que sigue
train_b, test_b = serie.iloc[:T_TRAIN], serie.iloc[T_TRAIN:T_TRAIN + HB]

arima_b = ajustar_arima(train_b, order_elegido)["modelo"]
pred_arima_b = np.asarray(arima_b.forecast(steps=HB))

pred_prophet_b = pred_prophet_safe(train_b, HB)
pred_naive_b = np.repeat(train_b.values[-1], HB)

mae_arima_b   = float(np.mean(np.abs(pred_arima_b - test_b.values)))
mae_naive_b   = float(np.mean(np.abs(pred_naive_b - test_b.values)))
mae_prophet_b = float(np.mean(np.abs(pred_prophet_b - test_b.values))) if pred_prophet_b is not None else float("nan")

plt.figure(figsize=(12, 5))
plt.plot(serie.index[:T_TRAIN + HB + 5], serie.values[:T_TRAIN + HB + 5], color="steelblue", label="Real", lw=1)
plt.plot(test_b.index, pred_arima_b,   color="crimson", ls="--", marker="o", ms=3, label=f"ARIMA{order_elegido} (MAE {mae_arima_b:,.0f})")
if pred_prophet_b is not None:
    plt.plot(test_b.index, pred_prophet_b, color="green", ls="--", marker="s", ms=3, label=f"Prophet (MAE {mae_prophet_b:,.0f})")
plt.plot(test_b.index, pred_naive_b,   color="gray",    ls=":",  marker="^", ms=3, label=f"Naive (MAE {mae_naive_b:,.0f})")
plt.axvline(test_b.index[0], color="gray", lw=0.8)
plt.title(f"Backtest B - caída post-pico - {CANCION_VIRAL}")
plt.xlabel("Fecha"); plt.ylabel("streams"); plt.legend()
plt.tight_layout(); plt.show()

prophet_txt_b = f"{mae_prophet_b:,.0f}" if not np.isnan(mae_prophet_b) else "no disponible"
print(f"Rango real del test B ({HB} días): mín {test_b.min():,.0f} | máx {test_b.max():,.0f} "
      f"| amplitud {test_b.max() - test_b.min():,.0f}")
print("-" * 60)
print(f"MAE Naive (último valor): {mae_naive_b:,.0f}")
print(f"MAE ARIMA{order_elegido}       : {mae_arima_b:,.0f}")
print(f"MAE Prophet             : {prophet_txt_b}")

**Backtest B - ¿alguien le gana al naive con dinámica real?** No. Sobre el tramo de
caída empinada (real de ~106k a ~73k, amplitud ~33k), **ARIMA(1,1,1) vuelve a empatar al
naive** (MAE ~16.760 vs ~16.748): como es un random-walk **sin término de deriva**,
pronostica una línea plana en el último valor de train y "se queda arriba", arrastrando la
bajada exactamente igual que el naive. **Prophet colapsa** (MAE ~511.800): con apenas 20
puntos de entrenamiento, su tendencia con changepoints + estacionalidades extrapola de
forma disparatada. Conclusión del tramo dinámico: **ninguno de los dos modelos le gana al
naive**; para batir a la persistencia en la caída haría falta un modelo que capture
explícitamente la **deriva** (p. ej. ARIMA con término de tendencia, o modelar el
decaimiento exponencial / log-streams).

## 11. Nota: por qué no redes neuronales

Se consideró usar una **red neuronal recurrente (RNN/LSTM)** para el pronóstico, pero se
descartó para este alcance: con ~200 puntos diarios **no hay volumen de datos suficiente**
para entrenar una red sin sobreajustar, y modelos clásicos como ARIMA/Prophet aprovechan
mejor una muestra chica. Una **CNN** tampoco aplica: no hay estructura espacial (ni de
vecindad tipo imagen) en una serie temporal univariada como esta.

## 12. Conclusión general

Los dos backtests, leídos contra el **baseline naive**, dan el hallazgo central:

| Tramo de test | Naive | ARIMA(1,1,1) | Prophet | Amplitud real |
|---|---|---|---|---|
| **A · meseta final** (últimos 25 d) | 2.358 | **2.358** | 106.595 | ~5.700 (~10%) |
| **B · caída post-pico** (días 20-44) | 16.748 | **16.760** | 511.797 | ~33.000 (~30%) |

**En la meseta, cualquier método (incluso el naive) funciona igual** porque la serie ya
está plana: el MAE bajo de ARIMA no reflejaba habilidad, sino la cola estable. **En la
subida/caída, donde SÍ hay dinámica, ARIMA(1,1,1) tampoco le gana al naive** (empata,
~16.750): al ser un random-walk sin deriva pronostica un valor constante y arrastra la
bajada igual que la persistencia, mientras que **Prophet empeora todo** (sobre-extrapola
con pocos datos). Dicho corto: **para esta serie viral, ni ARIMA(1,1,1) ni Prophet aportan
nada sobre el naive; Prophet directamente perjudica.**

- La parte metodológica es correcta y se sostiene: no estacionaria → `d=1` (ADF p≈0.001,
  KPSS p≈0.10 tras diferenciar), órdenes elegidos a mano por ACF/PACF y seleccionados por
  AIC → **ARIMA(1,1,1)**.
- Pero el valor agregado de los modelos sofisticados es **nulo o negativo** aquí. Para
  batir al naive en la caída haría falta capturar explícitamente la **deriva** (ARIMA con
  término de tendencia, o modelar el decaimiento exponencial / log-streams). Prophet recién
  sería razonable con series largas y estacionalidad anual real.
- **Qué usaríamos:** el **naive** como referencia obligada, y ARIMA(1,1,1) solo si se lo
  extiende con deriva; Prophet queda descartado para este tipo de serie corta y transitoria.